# 06 · OCR 面板 ROI 调参

只调 **绿色框**（中间「第 N 轮」+ 各品质总占位文字）。裁切结果在右侧预览，并可跑 OCR 看识别文本。

红色战利品格子框已默认关闭（那是未来「认格子」用，与当前 OCR 无关）。

满意后把 `CROP` 写回 `src/bidking_lab/capture/screen.py` 的 `INFO_PANEL_CROP_FRAC`。

In [ ]:
import sys
from pathlib import Path

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "scripts"))

from IPython.display import Image, display
from bidking_lab.capture.screen import INFO_PANEL_CROP_FRAC
from preview_panel_roi import render_roi_preview, ocr_on_crop

SAMPLE = REPO / "data" / "samples" / "panel_round4_1920x1080.png"
OUT = REPO / "data" / "samples" / "panel_round4_roi_preview.png"

print("仓库默认 ROI:", INFO_PANEL_CROP_FRAC)
print("样例图:", SAMPLE)

## 1. 改这里试裁剪（L, T, R, B 为相对整图的比例 0~1）

- **L 变大**：框右移，少裁左侧玩家列表  
- **R 变小**：框变窄，少吃进右侧仓库格  
- **T / B**：上下边界（避开顶栏、底部出价条）

In [ ]:
CROP = (0.30, 0.07, 0.59, 0.72)  # ← 改这四个数，重新运行本 cell 及以下

render_roi_preview(SAMPLE, crop=CROP, out_path=OUT, show_warehouse_hint=False)
display(Image(filename=str(OUT)))

## 2. 看 OCR 实际读到的字（与 Streamlit 同路径）

In [ ]:
text, err = ocr_on_crop(SAMPLE, CROP)
if err:
    print("错误:", err)
else:
    print(text)
    # 期望能看到：第4轮、蓝色35格、白绿12格、紫色34格、金色28格 等

## 3. 滑块微调（需 `pip install ipywidgets`）

In [ ]:
try:
    import ipywidgets as w
except ImportError:
    print("可选: pip install ipywidgets")
else:
    l, t, r, b = CROP

    @w.interact(
        L=(0.10, 0.45, 0.01),
        T=(0.0, 0.15, 0.01),
        R=(0.45, 0.70, 0.01),
        B=(0.55, 0.85, 0.01),
    )
    def tune(L=l, T=t, R=r, B=b):
        frac = (L, T, R, B)
        tmp = REPO / "data" / "samples" / "_roi_tune.png"
        render_roi_preview(SAMPLE, crop=frac, out_path=tmp, show_warehouse_hint=False)
        display(Image(filename=str(tmp)))
        txt, er = ocr_on_crop(SAMPLE, frac)
        print(f"crop={frac}")
        print(txt if not er else er)